In [1]:
# Importing Modules
import numpy as np
import pandas as pd
import itertools
from sklearn.model_selection import train_test_split
from src.utils.smiles2morganfp import MorganFP
from src.gnn_fp_utils.dataloader import loadData
from src.gnn_fp_utils.model import GNNModel
from src.train_test import TrainGNN, TestGNN
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Loading ESOL data
esol_data_train = pd.read_csv("data/train/ESOL.csv")
esol_data_val = pd.read_csv("data/val/ESOL.csv")

# Loading Lipophilicity data
lipophilicity_data_train = pd.read_csv("data/train/Lipophilicity.csv")
lipophilicity_data_val = pd.read_csv("data/val/Lipophilicity.csv")

# Loading FreeSolv data
freesolv_data_train = pd.read_csv("data/train/FreeSolv.csv")
freesolv_data_val = pd.read_csv("data/val/FreeSolv.csv")

# Loading B3DB data
b3db_data_train = pd.read_csv("data/train/B3DB.csv")
b3db_data_val = pd.read_csv("data/val/B3DB.csv")

# Loading RT data
rt_data_train = pd.read_csv("data/train/RT.csv")
rt_data_val = pd.read_csv("data/val/RT.csv")


In [3]:
# Function to run analysis
def RunGNN(data, dataName, modelName, h_dim, b_size, lr, d_out, wd, layers):

	train_data = data["train"].reset_index(drop=True)
	test_data = data["val"].reset_index(drop=True)

	train_fp = MorganFP(train_data["smiles"], bits=1024)
	train_fp["smiles"] = train_fp.index
	train_fp = train_fp.merge(train_data, on="smiles")

	test_fp = MorganFP(test_data["smiles"], bits=1024)
	test_fp["smiles"] = test_fp.index
	test_fp = test_fp.merge(test_data, on="smiles")

	# Loading dataset
	train_loader = loadData(train_fp, batch_size=b_size)
	test_loader = loadData(test_fp, batch_size=b_size)

	# Model
	model = GNNModel(num_features=118, hidden_dim=h_dim, model_type=modelName, dropout=d_out, num_layers=layers)

	# Model training
	model = TrainGNN(model, train_loader, test_loader, epochs=500, learning_rate=lr, w_decay=wd)

	# Model testing
	y_test, y_pred = TestGNN(model, test_loader)

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)

	# Return results
	return pd.DataFrame({"Data":[dataName], "Model":[modelName],
            "h_dim":[h_dim], "b_size":[b_size], "lr":[lr],
            "w_decay":[wd], "d_out":[d_out], "layers":[layers],
			"MAE":round(mae, 2), "RMSE":[round(rmse, 2)]})

In [4]:
# Function for model selection and hyperparameter tuning
def search_hyperparams(dataName):
    temp_out = []
    # Hyperparameter tuning loop
    for modelName in models:
        for i in grid_search_list:
            temp_out.append(RunGNN(data_dict[dataName], dataName, modelName, i["h_dim"], i["b_size"], i["lr"], i["dropout"], i["weight_decay"], i["layers"]))

    # Saving results
    final_df = pd.concat(temp_out, ignore_index=True)
    final_df.to_csv(f"results/Output_Hyperparameter_Optimization_GNN_FP_{dataName}.csv", quoting=False, index=False)

In [5]:
# Models
models = ["GCN", "GAT", "GIN", "GraphSAGE"]

# Hyperparameter dict
hyperparams = {
    'lr': [0.001],
    'h_dim': [32, 64],
    'b_size': [32],
    'weight_decay': [1e-4], 
    'dropout': [0.3],
    'layers': [2, 3]
}

# Data dict
data_dict = {
    "ESOL": {"train": esol_data_train, "val":esol_data_val}, 
    "Lipophilicity": {"train":lipophilicity_data_train, "val":lipophilicity_data_val},
    "RT": {"train":rt_data_train, "val":rt_data_val},
    "B3DB": {"train":b3db_data_train, "val": b3db_data_val},
    "FreeSolv":{"train":freesolv_data_train, "val":freesolv_data_val}
}

# Grid search list
keys = hyperparams.keys()
values = hyperparams.values()
grid_search_list = [dict(zip(keys, combination)) for combination in itertools.product(*values)]

In [ ]:
# Params Optim for ESOL
search_hyperparams("ESOL")

# Params Optim for Lipophilicity
search_hyperparams("Lipophilicity")

# Params Optim for FreeSolv
search_hyperparams("FreeSolv")

# Params Optim for RT
search_hyperparams("RT")

# Params Optim for B3DB
search_hyperparams("B3DB")